# 📘 Chapter 07 — Model Prediction & Evaluation 

## 🎯 Objectives  
In this chapter, we will load the optimized Random Forest model, generate predictions on the test set, evaluate model performance (MAE, RMSE, R²), visualize prediction quality, analyze residuals, and optionally export results.

### 🧩 Step 01 — Load Final Model & Test Set  

Load final rf model and access the processed train/test split.

In [4]:
import joblib
import pandas as pd

from src.config import MODEL_DIR
from src.utils.emoji_log import success, data, info

from src.modeling.train_baseline import (
    split_train_test,
    load_cleaned_data,
    build_features,
)
from src.features.feature_engineering import (
    clip_pollutants,
    add_rolling_features,
    handle_outliers_iqr,
    log_transform_features,
    scale_features,
)

In [5]:
# === 1. Data Preparation ===
df = load_cleaned_data("Taiwan")

# === 2. Feature Engineering Pipeline ===
df_clip = clip_pollutants(df.copy())
df_rolling = add_rolling_features(df_clip)
df_iqr = handle_outliers_iqr(df_rolling)
df_log = log_transform_features(df_iqr)

# === 3. Feature construction ===
X, y = build_features(df_log)

# === 4. Scaling (same method as model training) ===
X_scaled = scale_features(X, save_=False)

# === 5. Train-test split ===
data_split = split_train_test(X_scaled, y)

X_train = data_split["X_train"]
X_test = data_split["X_test"]
y_train = data_split["y_train"]
y_test = data_split["y_test"]

📂 Loading CSV: C:\Users\dinni\OneDrive\桌面\air_pollution\data\processed\Taiwan.csv
✅ Read CSV successfully! Shape: (5823862, 25)
✅ The pollutants limit has been set.
⚠️ Skipping nox due to NO + NO2 already exist.
✅ Rolling features added.
✅ IQR has been set.
✅ Pollutants skewes has been smoothed.
💬 Features shape: (5823862, 32), Target shape: (5823862,)
✅ Numerical features have been standardized. (32 columns scaled)
✅ Data successfully split! Train: (4659089, 32), Test: (1164773, 32)


In [3]:
# Load rf_final model
MODEL_PATH = MODEL_DIR / "rf_final_model_20251115_1215.pkl"
rf_model = joblib.load(MODEL_PATH)
success("Final model loaded successfully.")

✅ Final model loaded successfully.


### 🧮 Step 02 — Predict AQI on Test Data

Use the final Random Forest model to generate predictions.

In [7]:
y_pred_full = rf_model.predict(X_test)

In [8]:
# narrow down the amount of samples for visaulization
X_sample = X_test.sample(5000, random_state=42)
y_sample_actual = y_test.loc[X_sample.index]
y_sample_pred = rf_model.predict(X_sample)